# Week 08 — XML Disease Library Parsing
## LeafGuard AI: Building and Querying a Structured Disease Library

LeafGuard AI can store disease descriptions, symptoms, and treatments in XML. This notebook uses Python's XML tools to mirror what Android parsing code would do.

**Learning outcomes:**
- Generate well-formed XML content
- Validate XML structure with `ElementTree`
- Build a disease library dictionary from XML nodes
- Write search and filter helpers
- Connect XML parsing to the app's offline knowledge base

---


## Cell 1: Generate Sample `diseases.xml` Content (the real app stores this in `assets/diseases.xml`)

We begin with a small XML document that could live in the Android assets folder.


In [ ]:
disease_library_xml = """
<diseases>
  <disease id="tomato_early_blight" crop="Tomato" severity="medium">
    <name>Tomato Early Blight</name>
    <symptoms>Brown concentric rings on older leaves</symptoms>
    <treatment>Remove infected leaves and rotate crops</treatment>
  </disease>
  <disease id="tomato_late_blight" crop="Tomato" severity="high">
    <name>Tomato Late Blight</name>
    <symptoms>Dark lesions with rapid spread in humid conditions</symptoms>
    <treatment>Destroy infected plants and avoid overhead watering</treatment>
  </disease>
  <disease id="corn_rust" crop="Corn" severity="low">
    <name>Corn Common Rust</name>
    <symptoms>Small cinnamon-brown pustules on both leaf surfaces</symptoms>
    <treatment>Monitor spread and choose resistant varieties</treatment>
  </disease>
</diseases>
"""

print(disease_library_xml)


## Cell 2: Validate That the XML is Well-Formed

Before building features on top of XML, confirm that the document can be parsed without syntax errors.


In [ ]:
import xml.etree.ElementTree as ET

def validate_xml(xml_text):
    try:
        ET.fromstring(xml_text)
        return True, 'XML is well-formed.'
    except ET.ParseError as exc:
        return False, str(exc)

is_valid, message = validate_xml(disease_library_xml)
print(is_valid, message)


## Cell 3: Inspect the XML Tree

Each `<disease>` node contains attributes and child elements. This mirrors how Android's parsers walk through XML content.


In [ ]:
root = ET.fromstring(disease_library_xml)
print(f'Root tag: {root.tag}')
for disease in root.findall('disease'):
    print(
        disease.attrib['id'],
        '| crop=', disease.attrib['crop'],
        '| severity=', disease.attrib['severity'],
        '| name=', disease.findtext('name')
    )


## Cell 4: Build a Searchable Disease Library Dictionary

Converting XML into Python dictionaries makes downstream lookups easier.


In [ ]:
disease_library = {}
for disease in root.findall('disease'):
    disease_id = disease.attrib['id']
    disease_library[disease_id] = {
        'crop': disease.attrib['crop'],
        'severity': disease.attrib['severity'],
        'name': disease.findtext('name'),
        'symptoms': disease.findtext('symptoms'),
        'treatment': disease.findtext('treatment'),
    }

print('Library keys:', list(disease_library.keys()))
print('Sample entry:', disease_library['tomato_early_blight'])


## Cell 5: Implement Search and Filter Helpers

Search helpers make the XML data feel like a mini offline database.


In [ ]:
def search_by_crop(library, crop_name):
    return [entry for entry in library.values() if entry['crop'].lower() == crop_name.lower()]

def filter_by_severity(library, severity):
    return [entry for entry in library.values() if entry['severity'] == severity]

def keyword_search(library, keyword):
    keyword = keyword.lower()
    return [
        entry for entry in library.values()
        if keyword in entry['name'].lower() or keyword in entry['symptoms'].lower()
    ]

print('Tomato entries:', search_by_crop(disease_library, 'Tomato'))
print('High severity entries:', filter_by_severity(disease_library, 'high'))
print('Keyword "brown":', keyword_search(disease_library, 'brown'))


## Cell 6: Detect Missing Required Fields

Parsing is not the same as validation. A document may be well-formed yet still miss required data.


In [ ]:
required_fields = ['name', 'symptoms', 'treatment']
issues = []

for disease in root.findall('disease'):
    missing = [field for field in required_fields if not disease.findtext(field)]
    if missing:
        issues.append((disease.attrib.get('id', '<unknown>'), missing))

print('Validation issues:', issues if issues else 'None found')


## Summary and Quick Quiz

**Key takeaways:**
- XML is useful for structured offline content when a full database is unnecessary.
- Well-formedness and semantic validation are different checks.
- Converting XML nodes into dictionaries makes search functions simple.

**Quick quiz:**
1. What is the difference between well-formed XML and valid business data?
2. Why might an app choose XML for a disease library instead of hardcoded strings?
3. Which helper would you use to show only severe diseases?
4. How would you extend the schema to include prevention advice?
